
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 06</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Explore Version History and Time Travel</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">View a Delta table's change log and query data from any previous version.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

**Where we left off:** In Lessons 1 and 2, we created and modified the `employees` table with INSERT, UPDATE, and DELETE. Each operation created a new version. Let's start by looking at the current state of the table.

In [0]:
%sql
SELECT * 
FROM employees;


<!-- LEARN: Version History and Time Travel -->
<!-- Template: 2-card-colored-header-guidance -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Every Change is Recorded. Every Version is Queryable.</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Delta Lake automatically maintains a transaction log that records every operation performed on a table. You can inspect this log and query data as it existed at any point in the past.</div>

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Card 1: DESCRIBE HISTORY -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #4299E0; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">DESCRIBE HISTORY</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Shows the full change log for a table: every version number, timestamp, operation type, and who made the change.
    </div>
    <div style="background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> You need to audit what happened to a table, debug unexpected data, or find a version number to restore.
    </div>
  </div>
</div>

<!-- Card 2: Time Travel -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #00A972; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">VERSION AS OF / TIMESTAMP AS OF</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Query the table as it existed at a specific version number or point in time. The current table is unchanged; you're just reading the old data.
    </div>
    <div style="background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> You need to see what data looked like before a change, compare versions, or recover accidentally deleted rows.
    </div>
  </div>
</div>

</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**How Delta Lake versioning works**

- Every time you run a DML statement (INSERT, UPDATE, DELETE, MERGE) or a DDL statement (CREATE TABLE, ALTER TABLE), Delta Lake writes a new entry to the table's **transaction log**.
- Each entry is a **version**, starting at 0 (the initial CREATE TABLE) and incrementing by 1 for each subsequent operation.
- The transaction log records the operation type, timestamp, user, and which data files were added or removed.
- When you query a table normally (`SELECT * FROM employees`), you always get the latest version.
- Time travel lets you read any previous version by adding `VERSION AS OF <number>` or `TIMESTAMP AS OF '<datetime>'` to your query.
- Old versions are retained based on the table's retention policy (default: 30 days). After that, the underlying data files may be cleaned up by the `VACUUM` command.

**Why this matters**

- **Auditing:** You can prove exactly what changed, when, and who did it.
- **Debugging:** If a dashboard suddenly shows wrong numbers, you can compare the current version to yesterday's version to find the bad write.
- **Recovery:** If someone accidentally deletes rows, you can query the version before the delete and use it to restore the data.

</details>

### Explore: View the full version history

Let's inspect every change that's been made to the `employees` table since it was created.

In [0]:
%sql
DESCRIBE HISTORY employees;

You should see multiple versions, each with:
- **version** — the version number, starting at 0
- **timestamp** — when the operation happened
- **operation** — what type of change it was (CREATE TABLE, WRITE, UPDATE, DELETE)
- **userName** — who made the change

Take a moment to match each version to the operations you ran in Lesson 04.

### Explore: Query a previous version

Let's go back in time. Version 0 is the original table as it was created in Lesson 02, before any INSERT, UPDATE, or DELETE operations.

In [0]:
%sql
SELECT * 
FROM employees VERSION AS OF 0;

You should see just the **original 4 rows** from the CSV file. No Maria, no Aiden, no changes. This is exactly what the table looked like right after `CREATE TABLE` in Lesson 02.

Now let's check version 1, right after the INSERT (before the UPDATE and DELETE).

In [0]:
%sql
SELECT * 
FROM employees VERSION AS OF 1;

You should see **6 rows**: the original 4 plus Maria and Aiden. Maria's role is still `Data Engineer` (not yet updated) and Aiden is still present (not yet deleted).

### Explore: Compare versions side by side

A common pattern is comparing the current version to a previous one. Let's count how many rows exist in the current version versus version 0.

In [0]:
%sql
SELECT 'Current' AS version, COUNT(*) AS row_count FROM employees
UNION ALL
SELECT 'Version 0', COUNT(*) FROM employees VERSION AS OF 0;

This pattern is useful for debugging. If a table suddenly has more or fewer rows than expected, you can compare against a known-good version to identify which operation caused the change.

### Explore: Use the shorthand syntax

Databricks also supports a shorter syntax using `@v` followed by the version number.

In [0]:
%sql
SELECT * 
FROM employees@v0;

This returns the same result as `VERSION AS OF 0`. Use whichever syntax you find more readable.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Viewed the full change log with <code>DESCRIBE HISTORY</code></li>
      <li>Queried previous versions with <code>VERSION AS OF</code></li>
      <li>Compared row counts across versions to detect changes</li>
      <li>Used the <code>@v</code> shorthand syntax for time travel</li>
    </ul>
  </div>
</div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>